In [1]:
!pip install -q vllm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [2]:
import importlib.metadata as md
import torch

print("vllm :", md.version("vllm"))
print("torch:", torch.__version__)
print("cuda :", torch.cuda.is_available(), torch.cuda.get_device_name(0))

vllm : 0.33.0
torch: 2.14.0+cu130
cuda : True Tesla T4


In [3]:
import subprocess, time, requests

log = open("vllm.log", "w")
server = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-0.5B-Instruct",
    "--dtype", "half",
    "--max-model-len", "2048",
    "--gpu-memory-utilization", "0.85",
    "--max-num-seqs", "64",
    "--enable-prefix-caching",
    "--port", "8000",
], stdout=log, stderr=subprocess.STDOUT)

for _ in range(120):
    try:
        if requests.get("http://localhost:8000/health").status_code == 200:
            print("server ready"); break
    except Exception:
        pass
    time.sleep(5)
else:
    print(open("vllm.log").read()[-3000:])

server ready


In [9]:
import asyncio, aiohttp, time, numpy as np

URL = "http://localhost:8000/v1/chat/completions"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
PROMPT = "Explain the roofline model in simple words. " * 20

async def one(session):
    t0 = time.perf_counter(); ttft = None; n = 0
    body = {"model": MODEL, "messages": [{"role": "user", "content": PROMPT}],
            "max_tokens": 128, "stream": True}
    async with session.post(URL, json=body) as r:
        async for line in r.content:
            if line.startswith(b"data:") and b"[DONE]" not in line:
                if ttft is None: ttft = time.perf_counter() - t0
                n += 1
    return ttft, time.perf_counter() - t0, n

async def run(conc):
    total = conc * 10
    sem = asyncio.Semaphore(conc)
    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=600)) as s:
        async def guarded():
            async with sem: return await one(s)
        t0 = time.perf_counter()
        res = await asyncio.gather(*[guarded() for _ in range(total)])
        wall = time.perf_counter() - t0
    ttft = np.array([r[0] for r in res]); lat = np.array([r[1] for r in res])
    toks = sum(r[2] for r in res)
    print(f"conc={conc:4d}  rps={total/wall:6.1f}  tok/s={toks/wall:7.0f}  "
          f"TTFT p99={np.percentile(ttft,99):.2f}s  latency p99={np.percentile(lat,99):.2f}s")

await run(1)  # warmup
for c in [1, 4, 16, 64, 128, 256]:
    await run(c)

conc=   1  rps=   1.2  tok/s=    158  TTFT p99=0.04s  latency p99=0.86s
conc=   1  rps=   1.2  tok/s=    157  TTFT p99=0.03s  latency p99=0.82s
conc=   4  rps=   4.6  tok/s=    592  TTFT p99=0.08s  latency p99=0.89s
conc=  16  rps=  13.0  tok/s=   1648  TTFT p99=0.09s  latency p99=1.25s
conc=  64  rps=  27.2  tok/s=   3465  TTFT p99=0.20s  latency p99=2.46s
conc= 128  rps=  28.8  tok/s=   3652  TTFT p99=4.66s  latency p99=6.81s
conc= 256  rps=  28.6  tok/s=   3640  TTFT p99=11.54s  latency p99=13.75s


In [5]:
import re
txt = requests.get("http://localhost:8000/metrics").text
for l in txt.splitlines():
    if re.match(r"vllm:(num_requests_(running|waiting)|.*cache_usage_perc)", l):
        print(l)

vllm:num_requests_running{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0
vllm:num_requests_waiting{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0
vllm:num_requests_waiting_by_reason{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct",reason="capacity"} 0.0
vllm:num_requests_waiting_by_reason{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct",reason="deferred"} 0.0
vllm:kv_cache_usage_perc{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0


In [6]:
import re

async def sample_metrics(stop, rows):
    async with aiohttp.ClientSession() as s:
        while not stop.is_set():
            txt = await (await s.get("http://localhost:8000/metrics")).text()
            vals = {}
            for l in txt.splitlines():
                m = re.match(r'vllm:(num_requests_running|num_requests_waiting|kv_cache_usage_perc)\{[^}]*\}\s+([\d.eE+-]+)', l)
                if m: vals[m.group(1)] = float(m.group(2))
            rows.append(vals)
            await asyncio.sleep(0.5)

async def run_with_metrics(conc):
    stop, rows = asyncio.Event(), []
    task = asyncio.create_task(sample_metrics(stop, rows))
    await run(conc)
    stop.set(); await task
    g = lambda k: max(r.get(k, 0) for r in rows) if rows else 0
    print(f"      max running={g('num_requests_running'):.0f}  "
          f"max waiting={g('num_requests_waiting'):.0f}  "
          f"max KV={100*g('kv_cache_usage_perc'):.1f}%")

for c in [24, 32, 48, 64, 128]:
    await run_with_metrics(c)

conc=  24  rps=  16.7  tok/s=   2127  TTFT p99=0.14s  latency p99=1.52s
      max running=24  max waiting=0  max KV=0.3%
conc=  32  rps=  20.2  tok/s=   2597  TTFT p99=0.12s  latency p99=1.59s
      max running=32  max waiting=0  max KV=0.5%
conc=  48  rps=  24.0  tok/s=   3061  TTFT p99=0.18s  latency p99=2.04s
      max running=48  max waiting=0  max KV=0.6%
conc=  64  rps=  27.1  tok/s=   3437  TTFT p99=0.22s  latency p99=2.43s
      max running=64  max waiting=0  max KV=0.9%
conc= 128  rps=  27.4  tok/s=   3503  TTFT p99=4.81s  latency p99=7.07s
      max running=64  max waiting=36  max KV=0.8%


In [7]:
import uuid

async def one(session):
    t0 = time.perf_counter(); ttft = None; n = 0
    content = f"Request {uuid.uuid4()}. " + PROMPT
    body = {"model": MODEL, "messages": [{"role": "user", "content": content}],
            "max_tokens": 128, "stream": True}
    async with session.post(URL, json=body) as r:
        async for line in r.content:
            if line.startswith(b"data:") and b"[DONE]" not in line:
                if ttft is None: ttft = time.perf_counter() - t0
                n += 1
    return ttft, time.perf_counter() - t0, n

In [8]:
for c in [16, 24, 32, 40, 48, 64]:
    await run_with_metrics(c)

conc=  16  rps=  11.4  tok/s=   1440  TTFT p99=0.36s  latency p99=1.55s
      max running=16  max waiting=0  max KV=0.6%
conc=  24  rps=  13.9  tok/s=   1777  TTFT p99=0.40s  latency p99=1.84s
      max running=24  max waiting=0  max KV=0.9%
conc=  32  rps=  15.7  tok/s=   2010  TTFT p99=0.54s  latency p99=2.22s
      max running=32  max waiting=0  max KV=1.2%
conc=  40  rps=  16.6  tok/s=   2134  TTFT p99=0.66s  latency p99=2.72s
      max running=40  max waiting=3  max KV=1.5%
conc=  48  rps=  17.4  tok/s=   2232  TTFT p99=0.82s  latency p99=3.22s
      max running=48  max waiting=11  max KV=1.8%
conc=  64  rps=  19.5  tok/s=   2496  TTFT p99=1.07s  latency p99=3.93s
      max running=64  max waiting=27  max KV=2.3%


In [12]:
import uuid, asyncio, aiohttp, time, numpy as np

async def one(session):
    t0 = time.perf_counter(); ttft = None; n = 0
    content = f"Request {uuid.uuid4()}. " + PROMPT
    body = {"model": MODEL, "messages": [{"role": "user", "content": content}],
            "max_tokens": 128, "stream": True}
    async with session.post(URL, json=body) as r:
        async for line in r.content:
            if line.startswith(b"data:") and b"[DONE]" not in line:
                if ttft is None: ttft = time.perf_counter() - t0
                n += 1
    return ttft, time.perf_counter() - t0, n

async def run(conc):
    total = conc * 10
    sem = asyncio.Semaphore(conc)
    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=600)) as s:
        async def guarded():
            async with sem: return await one(s)
        t0 = time.perf_counter()
        res = await asyncio.gather(*[guarded() for _ in range(total)])
        wall = time.perf_counter() - t0
    ttft = np.array([r[0] for r in res]); lat = np.array([r[1] for r in res])
    toks = sum(r[2] for r in res)
    print(f"conc={conc:4d}  n={total:4d}  rps={total/wall:6.1f}  tok/s={toks/wall:7.0f}  "
          f"TTFT p99={np.percentile(ttft,99):.2f}s  latency p99={np.percentile(lat,99):.2f}s")

In [14]:
import subprocess, statistics

async def run_with_gpu(conc):
    p = subprocess.Popen(
        ["nvidia-smi", "--query-gpu=utilization.gpu,power.draw,clocks.sm",
         "--format=csv,noheader,nounits", "-lms", "500"],
        stdout=subprocess.PIPE, text=True)
    await run(conc)
    p.terminate()
    rows = [l.strip().split(", ") for l in p.stdout.read().splitlines() if l.strip()]
    rows = [r for r in rows if len(r) == 3]
    util = [float(r[0]) for r in rows]; pw = [float(r[1]) for r in rows]; clk = [float(r[2]) for r in rows]
    print(f"   GPU util median={statistics.median(util):.0f}% max={max(util):.0f}%  "
          f"power median={statistics.median(pw):.0f}W  SM clock median={statistics.median(clk):.0f}MHz")

for c in [1, 16, 32, 64]:
    await run_with_gpu(c)

conc=   1  n=  10  rps=   1.2  tok/s=    155  TTFT p99=0.05s  latency p99=0.88s
   GPU util median=0% max=100%  power median=28W  SM clock median=442MHz
conc=  16  n= 160  rps=  11.3  tok/s=   1448  TTFT p99=0.27s  latency p99=1.46s
   GPU util median=0% max=100%  power median=27W  SM clock median=660MHz
conc=  32  n= 320  rps=  15.4  tok/s=   1974  TTFT p99=0.55s  latency p99=2.26s
   GPU util median=0% max=100%  power median=28W  SM clock median=638MHz
conc=  64  n= 640  rps=  19.9  tok/s=   2555  TTFT p99=1.04s  latency p99=3.87s
   GPU util median=0% max=100%  power median=28W  SM clock median=660MHz


In [13]:
for c in [1, 16, 32, 64]:
    await run_with_gpu(c)

NameError: name 'run_with_gpu' is not defined

In [15]:
import os, psutil, asyncio, numpy as np

me = psutil.Process(os.getpid())

def server_procs():
    out = []
    for p in psutil.process_iter(["name", "cmdline"]):
        s = (p.info["name"] or "") + " ".join(p.info["cmdline"] or [])
        if "vllm" in s.lower() and p.pid != os.getpid():
            out.append(p)
    return out

async def sample_host(stop, rows, procs):
    for p in procs: p.cpu_percent(None)
    me.cpu_percent(None)
    while not stop.is_set():
        await asyncio.sleep(0.5)
        proc = await asyncio.create_subprocess_exec(
            "nvidia-smi", "--query-gpu=utilization.gpu,power.draw,clocks.sm",
            "--format=csv,noheader,nounits", stdout=asyncio.subprocess.PIPE)
        out, _ = await proc.communicate()
        u, pw, ck = [float(x) for x in out.decode().strip().split(", ")]
        srv = max((p.cpu_percent(None) for p in procs), default=0)
        rows.append((u, pw, ck, srv, me.cpu_percent(None)))

async def run_with_host(conc):
    procs, stop, rows = server_procs(), asyncio.Event(), []
    task = asyncio.create_task(sample_host(stop, rows, procs))
    await run(conc)
    stop.set(); await task
    a = np.array(rows)
    print(f"   GPU util median={np.median(a[:,0]):.0f}% p90={np.percentile(a[:,0],90):.0f}%  "
          f"power={np.median(a[:,1]):.0f}W  clock={np.median(a[:,2]):.0f}MHz  "
          f"busiest vLLM process CPU={np.median(a[:,3]):.0f}%  notebook CPU={np.median(a[:,4]):.0f}%  "
          f"(cores={psutil.cpu_count()})")

for c in [1, 16, 32, 64]:
    await run_with_host(c)

conc=   1  n=  10  rps=   1.2  tok/s=    155  TTFT p99=0.05s  latency p99=0.89s


ValueError: could not convert string to float: '1095\n0'

In [16]:
!nvidia-smi -L
!nvidia-smi --query-gpu=index,memory.used --format=csv

GPU 0: Tesla T4 (UUID: GPU-e523c599-fdc0-9679-5f7b-b71e62108dea)
GPU 1: Tesla T4 (UUID: GPU-4a21ee29-b268-aa2f-4eef-8996f65b5745)
index, memory.used [MiB]
0, 12655 MiB
1, 3 MiB


In [18]:
async def sample_host(stop, rows, procs):
    for p in procs: p.cpu_percent(None)
    me.cpu_percent(None)
    while not stop.is_set():
        await asyncio.sleep(0.5)
        proc = await asyncio.create_subprocess_exec(
            "nvidia-smi", "-i", "0",
            "--query-gpu=utilization.gpu,power.draw,clocks.sm",
            "--format=csv,noheader,nounits", stdout=asyncio.subprocess.PIPE)
        out, _ = await proc.communicate()
        line = out.decode().strip().splitlines()[0]
        u, pw, ck = [float(x) for x in line.split(", ")]
        srv = max((p.cpu_percent(None) for p in procs), default=0)
        rows.append((u, pw, ck, srv, me.cpu_percent(None)))

In [19]:
for c in [1, 16, 32, 64]:
    await run_with_host(c)

conc=   1  n=  10  rps=   1.2  tok/s=    154  TTFT p99=0.05s  latency p99=0.90s
   GPU util median=100% p90=100%  power=66W  clock=1042MHz  busiest vLLM process CPU=61%  notebook CPU=2%  (cores=4)
conc=  16  n= 160  rps=  11.2  tok/s=   1441  TTFT p99=0.27s  latency p99=1.46s
   GPU util median=100% p90=100%  power=66W  clock=1125MHz  busiest vLLM process CPU=54%  notebook CPU=13%  (cores=4)
conc=  32  n= 320  rps=  15.3  tok/s=   1960  TTFT p99=0.55s  latency p99=2.27s
   GPU util median=100% p90=100%  power=65W  clock=1080MHz  busiest vLLM process CPU=46%  notebook CPU=16%  (cores=4)
conc=  64  n= 640  rps=  20.0  tok/s=   2552  TTFT p99=1.05s  latency p99=3.91s
   GPU util median=100% p90=100%  power=66W  clock=1140MHz  busiest vLLM process CPU=43%  notebook CPU=20%  (cores=4)


In [20]:
import os
env = dict(os.environ, CUDA_VISIBLE_DEVICES="1")
log2 = open("vllm2.log", "w")
server2 = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-0.5B-Instruct", "--dtype", "half",
    "--max-model-len", "2048", "--gpu-memory-utilization", "0.85",
    "--max-num-seqs", "64", "--enable-prefix-caching", "--port", "8001",
], env=env, stdout=log2, stderr=subprocess.STDOUT)

for _ in range(120):
    try:
        if requests.get("http://localhost:8001/health").status_code == 200:
            print("server 2 ready"); break
    except Exception:
        pass
    time.sleep(5)
else:
    print(open("vllm2.log").read()[-3000:])

server 2 ready


In [21]:
import itertools
URLS = ["http://localhost:8000/v1/chat/completions",
        "http://localhost:8001/v1/chat/completions"]
rr = itertools.count()

async def one(session):
    url = URLS[next(rr) % len(URLS)]
    t0 = time.perf_counter(); ttft = None; n = 0
    content = f"Request {uuid.uuid4()}. " + PROMPT
    body = {"model": MODEL, "messages": [{"role": "user", "content": content}],
            "max_tokens": 128, "stream": True}
    async with session.post(url, json=body) as r:
        async for line in r.content:
            if line.startswith(b"data:") and b"[DONE]" not in line:
                if ttft is None: ttft = time.perf_counter() - t0
                n += 1
    return ttft, time.perf_counter() - t0, n

for c in [32, 48, 64, 128]:
    await run(c)

conc=  32  n= 320  rps=  20.0  tok/s=   2565  TTFT p99=2.28s  latency p99=3.83s
conc=  48  n= 480  rps=  26.7  tok/s=   3433  TTFT p99=0.43s  latency p99=2.07s
conc=  64  n= 640  rps=  30.6  tok/s=   3920  TTFT p99=0.56s  latency p99=2.49s
conc= 128  n=1280  rps=  36.8  tok/s=   4710  TTFT p99=25.51s  latency p99=26.94s


In [22]:
async def run(conc):
    total = conc * 10
    sem = asyncio.Semaphore(conc)
    conn = aiohttp.TCPConnector(limit=0)
    async with aiohttp.ClientSession(connector=conn,
            timeout=aiohttp.ClientTimeout(total=600)) as s:
        async def guarded():
            async with sem: return await one(s)
        t0 = time.perf_counter()
        res = await asyncio.gather(*[guarded() for _ in range(total)])
        wall = time.perf_counter() - t0
    ttft = np.array([r[0] for r in res]); lat = np.array([r[1] for r in res])
    toks = sum(r[2] for r in res)
    print(f"conc={conc:4d}  n={total:4d}  rps={total/wall:6.1f}  tok/s={toks/wall:7.0f}  "
          f"TTFT p99={np.percentile(ttft,99):.2f}s  latency p99={np.percentile(lat,99):.2f}s")

await run(2)   # warmup: hits both servers
for c in [32, 40, 48, 64, 96, 128, 160]:
    await run(c)

conc=   2  n=  20  rps=   2.4  tok/s=    305  TTFT p99=0.05s  latency p99=0.91s
conc=  32  n= 320  rps=  22.1  tok/s=   2835  TTFT p99=0.30s  latency p99=1.57s
conc=  40  n= 400  rps=  24.5  tok/s=   3152  TTFT p99=0.35s  latency p99=1.79s
conc=  48  n= 480  rps=  27.5  tok/s=   3531  TTFT p99=0.44s  latency p99=1.94s
conc=  64  n= 640  rps=  31.6  tok/s=   4048  TTFT p99=0.71s  latency p99=2.34s
conc=  96  n= 960  rps=  36.4  tok/s=   4666  TTFT p99=0.78s  latency p99=2.99s
conc= 128  n=1280  rps=  40.3  tok/s=   5161  TTFT p99=1.07s  latency p99=3.63s
conc= 160  n=1600  rps=  40.0  tok/s=   5136  TTFT p99=3.48s  latency p99=6.52s
